In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
labels = ['PNEUMONIA','NORMAL']
img_size = 150

def get_data(dir):
    data = []
    for label in labels:
        path = os.path.join(dir, label)
        class_num = labels.index(label)

        for img in os.listdir(path):
            if img.endswith('jpeg'):
                img_arr = cv2.imread(os.path.join(path,img), cv2.IMREAD_GRAYSCALE)
                resized_arr = cv2.resize(img_arr, (img_size, img_size))
                data.append([resized_arr, class_num])

    return np.array(data, dtype='object')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
train = get_data('/content/drive/MyDrive/data_set/chest_xray/train')
val   = get_data('/content/drive/MyDrive/data_set/chest_xray/val')
test  = get_data('/content/drive/MyDrive/data_set/chest_xray/test')

In [ ]:
def split_data(data):
    X, y = [], []
    for feature, label in data:
        X.append(feature)
        y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = split_data(train)
X_val, y_val     = split_data(val)
X_test, y_test   = split_data(test)

In [ ]:
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0

In [ ]:
X_train = tf.image.resize(X_train[..., np.newaxis], (224,224))
X_val   = tf.image.resize(X_val[..., np.newaxis], (224,224))
X_test  = tf.image.resize(X_test[..., np.newaxis], (224,224))

X_train = tf.image.grayscale_to_rgb(X_train)
X_val   = tf.image.grayscale_to_rgb(X_val)
X_test  = tf.image.grayscale_to_rgb(X_test)

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))
print(class_weights)

{np.int64(0): np.float64(0.5316195372750643), np.int64(1): np.float64(8.40650406504065)}


In [ ]:
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,
    factor=0.3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=15,
    class_weight=class_weights,
    callbacks=[lr_reduce, early_stop]
)

Epoch 1/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 109s 611ms/step - accuracy: 0.7553 - loss: 0.5290 - val_accuracy: 0.8750 - val_loss: 0.3722 - learning_rate: 1.0000e-04
Epoch 2/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 53s 403ms/step - accuracy: 0.8503 - loss: 0.3338 - val_accuracy: 0.6250 - val_loss: 0.6235 - learning_rate: 1.0000e-04
Epoch 3/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 53s 411ms/step - accuracy: 0.8677 - loss: 0.2940 - val_accuracy: 0.7500 - val_loss: 0.5608 - learning_rate: 1.0000e-04
Epoch 4/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - accuracy: 0.8827 - loss: 0.2671
Epoch 4: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
130/130 ━━━━━━━━━━━━━━━━━━━━ 53s 405ms/step - accuracy: 0.8868 - loss: 0.2539 - val_accuracy: 0.7500 - val_loss: 0.4080 - learning_rate: 1.0000e-04
Epoch 5/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 54s 418ms/step - accuracy: 0.8958 - loss: 0.2304 - val_accuracy: 0.8750 - val_loss: 0.3626 - learning_rate: 3.0000e-05
Epoch 6/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 53s 404ms/

In [ ]:
model.evaluate(X_test, y_test)

20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 732ms/step - accuracy: 0.8622 - loss: 0.3639


[0.3639024794101715, 0.8621794581413269]

In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

print(classification_report(y_test, y_pred))

20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 768ms/step
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       390
           1       0.77      0.89      0.83       234

    accuracy                           0.86       624
   macro avg       0.85      0.87      0.86       624
weighted avg       0.87      0.86      0.86       624



In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    img = cv2.resize(img, (224, 224))
    img = img / 255.0

    img = np.expand_dims(img, axis=-1)   # (224,224,1)


    img = tf.convert_to_tensor(img, dtype=tf.float32)

    img = tf.image.grayscale_to_rgb(img) # now works

    img = tf.expand_dims(img, axis=0)    # (1,224,224,3)

    return img

In [ ]:
img = preprocess_image("/content/p.jpg")

prediction = model.predict(img)[0][0]

if prediction > 0.5:
    print("NORMAL")
else:
    print("PNEUMONIA")

print("Confidence:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
PNEUMONIA
Confidence: 0.0062558595


In [ ]:
img = preprocess_image("/content/norm.jpg")

prediction = model.predict(img)[0][0]

if prediction > 0.5:
    print("NORMAL")
else:
    print("PNEUMONIA")

print("Confidence:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
NORMAL
Confidence: 0.6168192


In [38]:
model.save("pneumonia_model.h5")